In [1]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pybedtools

# ---------- I/O helpers ----------

def read_fragments(path):
    """
    fragments.tsv with at least 4 cols:
      chr  start  end  barcode 
    """
    df = pd.read_csv(path, sep="\t", header=None, comment="#", usecols=[0,1,2,3],
                     names=["chr","start","end","barcode"], engine="c")
    df["chr"] = df["chr"].astype(str)
    df["chr"] = np.where(df["chr"].str.startswith("chr"), df["chr"], "chr" + df["chr"])
    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)
    df["barcode"] = df["barcode"].astype(str)
    return df

def make_windows(chrom_sizes_path, w=50):
    """Return a DataFrame of fixed windows (chr, start, end, win_id)."""
    sizes = pd.read_csv(chrom_sizes_path, sep="\t", header=None, names=["chr","size"])
    sizes["chr"] = sizes["chr"].astype(str)
    rows = []
    for _, r in sizes.iterrows():
        chrn = r["chr"]
        size = int(r["size"])
        starts = np.arange(0, size, w, dtype=np.int64)
        ends = np.minimum(starts + w, size).astype(np.int64)
        rows.append(pd.DataFrame({"chr": chrn, "start": starts, "end": ends}))
    win = pd.concat(rows, ignore_index=True)
    win["win_id"] = np.arange(len(win), dtype=np.int64)
    return win

In [2]:
bitchass = make_windows("../files_etc/dm6.chrom.sizes")

In [3]:
bitchass

,chr,start,end,win_id
0,chr2L,0,50,0
1,chr2L,50,100,1
2,chr2L,100,150,2
3,chr2L,150,200,3
4,chr2L,200,250,4
...,...,...,...,...
2650647,chrX,23542050,23542100,2650647
2650648,chrX,23542100,23542150,2650648
2650649,chrX,23542150,23542200,2650649
2650650,chrX,23542200,23542250,2650650


In [2]:
# ---------- core ----------

def cell_by_window_matrix(frag_df, win_df):
    """
    Build sparse matrix M (barcodes x windows) of overlap COUNTS.
    Each fragment counts as 1 (dupcount ignored).
    """
    win_bt  = pybedtools.BedTool.from_dataframe(win_df[["chr","start","end","win_id"]])
    frag_bt = pybedtools.BedTool.from_dataframe(frag_df[["chr","start","end","barcode"]])

    ov = win_bt.intersect(frag_bt, wa=True, wb=True)
    cols = ["w_chr","w_start","w_end","win_id","f_chr","f_start","f_end","barcode"]
    ov_df = ov.to_dataframe(names=cols, dtype={"win_id": np.int64, "barcode": str})

    # count overlaps per (barcode, window)
    g = ov_df.groupby(["barcode","win_id"], sort=False).size().reset_index(name="n")

    barcodes = g["barcode"].unique()
    b2i = {b:i for i,b in enumerate(barcodes)}
    n_rows, n_cols = len(barcodes), win_df.shape[0]

    row = g["barcode"].map(b2i).to_numpy()
    col = g["win_id"].to_numpy()
    dat = g["n"].to_numpy(dtype=np.float64)

    M = sp.coo_matrix((dat, (row, col)), shape=(n_rows, n_cols)).tocsr()
    return M, barcodes

def row_normalize_by_fragcount(M, frag_df, barcodes):
    """
    Divide each row by the number of fragments for that barcode
    (simple count of rows in fragments for that barcode; dupcount ignored).
    """
    counts = frag_df.groupby("barcode").size()  # Series barcode -> n_frags
    row_tot = np.array([counts.get(b, 0) for b in barcodes], dtype=np.float64)
    row_tot[row_tot == 0] = 1.0
    Dinv = sp.diags(1.0 / row_tot)
    return Dinv @ M

def build_matrices(fragments_path, chrom_sizes_path, win=50):
    frag = read_fragments(fragments_path)
    wins = make_windows(chrom_sizes_path, w=win)
    M, barcodes = cell_by_window_matrix(frag, wins)
    M_norm = row_normalize_by_fragcount(M, frag, barcodes)
    col_sums = np.asarray(M_norm.sum(axis=0)).ravel()
    return M, M_norm, col_sums, barcodes, wins

In [9]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pybedtools

try:
    import pyBigWig
    HAVE_PYBW = True
except Exception:
    HAVE_PYBW = False

# -------------------- helpers --------------------

def read_fragments_ignore_dupcount(path):
    # fragments.tsv (or .gz) with columns: chr start end barcode [dupcount...]
    df = pd.read_csv(path, sep="\t", header=None, comment="#", usecols=[0,1,2,3],
                     names=["chr","start","end","barcode"], engine="c")
    df["chr"] = df["chr"].astype(str)
    df["chr"] = np.where(df["chr"].str.startswith("chr"), df["chr"], "chr" + df["chr"])
    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)
    df["barcode"] = df["barcode"].astype(str)
    return df

def make_windows(chrom_sizes_path, w=50):
    sizes = pd.read_csv(chrom_sizes_path, sep="\t", header=None, names=["chr","size"])
    sizes["chr"] = sizes["chr"].astype(str)
    all_rows = []
    for _, r in sizes.iterrows():
        chrn = r["chr"]; size = int(r["size"])
        starts = np.arange(0, size, w, dtype=np.int64)
        ends = np.minimum(starts + w, size)
        all_rows.append(pd.DataFrame({"chr": chrn, "start": starts, "end": ends}))
    win = pd.concat(all_rows, ignore_index=True)
    win["win_id"] = np.arange(len(win), dtype=np.int64)
    return win, sizes

def cell_by_window_matrix(frag_df, win_df):
    # Build sparse (barcodes × windows) overlap counts (each fragment counts as 1)
    win_bt  = pybedtools.BedTool.from_dataframe(win_df[["chr","start","end","win_id"]])
    frag_bt = pybedtools.BedTool.from_dataframe(frag_df[["chr","start","end","barcode"]])
    ov = win_bt.intersect(frag_bt, wa=True, wb=True)
    cols = ["w_chr","w_start","w_end","win_id","f_chr","f_start","f_end","barcode"]
    ov_df = ov.to_dataframe(names=cols, dtype={"win_id": np.int64, "barcode": str})
    if ov_df.empty:
        # no overlaps -> return empty structures
        barcodes = np.array([], dtype=str)
        M = sp.csr_matrix((0, win_df.shape[0]), dtype=np.float64)
        return M, barcodes
    g = ov_df.groupby(["barcode","win_id"], sort=False).size().reset_index(name="n")

    barcodes = g["barcode"].unique()
    b2i = {b:i for i,b in enumerate(barcodes)}
    n_rows, n_cols = len(barcodes), win_df.shape[0]
    row = g["barcode"].map(b2i).to_numpy()
    col = g["win_id"].to_numpy()
    dat = g["n"].to_numpy(dtype=np.float64)
    M = sp.coo_matrix((dat, (row, col)), shape=(n_rows, n_cols)).tocsr()
    return M, barcodes

def row_normalize_by_fragcount(M, frag_df, barcodes):
    # Divide each row by number of fragments for that barcode (dupcount ignored)
    if M.shape[0] == 0:
        return M
    counts = frag_df.groupby("barcode").size()
    row_tot = np.array([counts.get(b, 0) for b in barcodes], dtype=np.float64)
    row_tot[row_tot == 0] = 1.0
    return sp.diags(1.0 / row_tot) @ M

def write_outputs(prefix, windows, values, chrom_sizes_df, make_bw=True):
    # BedGraph
    bg = pd.DataFrame({
        "chr": windows["chr"].to_numpy(),
        "start": windows["start"].to_numpy(),
        "end": windows["end"].to_numpy(),
        "score": values
    })
    bg_path = f"{prefix}.bedGraph"
    bg.to_csv(bg_path, sep="\t", header=False, index=False)

    # BigWig (optional)
    if make_bw and HAVE_PYBW:
        bw = pyBigWig.open(f"{prefix}.bw", "w")
        chrom_sizes = list(map(tuple, chrom_sizes_df.astype({ "size": int }).values.tolist()))
        bw.addHeader(chrom_sizes)
        # pyBigWig expects 0-based starts; we already have 0-based; do NOT add +1
        bw.addEntries(bg["chr"].tolist(), bg["start"].tolist(), ends=bg["end"].tolist(),
                      values=bg["score"].astype(float).tolist())
        bw.close()

# -------------------- driver --------------------

def build_cluster_maps(frags_csv, time_token):
    """
    Collect cluster -> set(barcodes) for all directories whose key contains time_token.
    Assumes those dirs contain one CSV per cluster with barcodes (no header).
    """
    from collections import defaultdict
    cluster2barcodes = defaultdict(set)
    dirs = [k for k in frags_csv if time_token in k]
    for d in dirs:
        for f in os.listdir(d):
            if not f.endswith(".csv"):
                continue
            cl = os.path.splitext(f)[0]
            bcs = pd.read_csv(os.path.join(d, f), header=None)[0].astype(str)
            for b in bcs:
                cluster2barcodes[cl].add(b)
    return cluster2barcodes

def collect_frag_files(frags_csv, time_token, modality_token):
    # Gather all fragment paths matching time and modality across replicates
    frag_files = []
    for k, paths in frags_csv.items():
        if time_token not in k:
            continue
        for p in paths:
            if modality_token in p:
                frag_files.append(p)
    return frag_files

def run_all(frags_csv, chrom_sizes_path, outdir, win=50,
            timepoints=("8_10h","14_16h"),
            modalities=("h3k27ac","h3k27me3"),
            scale=100000):   # <<< NEW: balancing coef
    os.makedirs(outdir, exist_ok=True)
    windows, chrom_sizes_df = make_windows(chrom_sizes_path, w=win)

    for tp in timepoints:
        cluster_map = build_cluster_maps(frags_csv, tp)  # cluster -> set(barcodes)
        if not cluster_map:
            continue
        for mod in modalities:
            frag_files = collect_frag_files(frags_csv, tp, mod)
            if not frag_files:
                continue
            for cl, bc_set in cluster_map.items():
                if not bc_set:
                    continue
                # read & filter fragments from all reps for this modality+time
                frag_parts = []
                for fp in frag_files:
                    df = read_fragments_ignore_dupcount(fp)
                    frag_parts.append(df[df["barcode"].isin(bc_set)])
                frag = pd.concat([x for x in frag_parts if not x.empty], ignore_index=True)
                if frag.empty:
                    continue

                # build cell×window, row-normalize, sum columns
                M, barcodes = cell_by_window_matrix(frag, windows)
                M_norm = row_normalize_by_fragcount(M, frag, barcodes)
                col_sums = np.asarray(M_norm.sum(axis=0)).ravel()

                # per-cell normalization
                n_cells = len(bc_set)  # or len(barcodes) if you prefer only cells with any reads
                if n_cells == 0:
                    continue
                col_sums_per_cell = col_sums / float(n_cells)

                # <<< NEW: apply balancing coefficient
                col_sums_scaled = col_sums_per_cell * float(scale)

                # write outputs
                safe_cl = cl.replace(" ", "_")
                prefix = os.path.join(outdir, f"{tp}__{mod}__{safe_cl}__win{win}")
                write_outputs(prefix, windows, col_sums_scaled, chrom_sizes_df, make_bw=True)

                with open(f"{prefix}.barcodes.txt","w") as fh:
                    for b in sorted(bc_set):
                        fh.write(b + "\n")


# -------------------- usage --------------------
# run_all(frags_csv, "dm6.chrom.sizes", outdir="pb_50nt_out", win=50)


In [6]:
import pickle

In [7]:
with open("../pickle/fragscsv_wt.pickle", 'rb') as f:
    frags_csv = pickle.load(f)


In [11]:
run_all(frags_csv, "../files_etc/dm6.chrom.sizes2", outdir="../pb_50nt_out", win = 50)

In [22]:
with open("../pickle/fragscsv_kd.pickle", 'rb') as f:
    frags_csv_kd = pickle.load(f)


In [25]:
run_all(frags_csv_kd, "../files_etc/dm6.chrom.sizes2", outdir="../pb_50nt_out_ezkd", win = 50)

In [23]:
frags_csv

{'/Users/artemilin/Work/projects/Landscapes/scCandT/pseudobulk_csv/batch1_14_16h': ['/Users/artemilin/Work/projects/Landscapes/scCandT/scct_fragments/rep1/emb14_16h_h3k27ac_1_fragments.tsv',
  '/Users/artemilin/Work/projects/Landscapes/scCandT/scct_fragments/rep1/emb14_16h_h3k27me3_1_fragments.tsv'],
 '/Users/artemilin/Work/projects/Landscapes/scCandT/pseudobulk_csv/batch1_8_10h': ['/Users/artemilin/Work/projects/Landscapes/scCandT/scct_fragments/rep1/emb8_10h_h3k27ac_1_fragments.tsv',
  '/Users/artemilin/Work/projects/Landscapes/scCandT/scct_fragments/rep1/emb8_10h_h3k27me3_1_fragments.tsv'],
 '/Users/artemilin/Work/projects/Landscapes/scCandT/pseudobulk_csv/batch2_14_16h': ['/Users/artemilin/Work/projects/Landscapes/scCandT/scct_fragments/rep2/emb_14_16h_h3k27ac_2_fragments.tsv',
  '/Users/artemilin/Work/projects/Landscapes/scCandT/scct_fragments/rep2/emb_14_16h_h3k27me3_2_fragments.tsv'],
 '/Users/artemilin/Work/projects/Landscapes/scCandT/pseudobulk_csv/batch2_8_10h': ['/Users/arte

In [24]:
frags_csv_kd

{'/Users/artemilin/Work/projects/Landscapes/scCandT/pseudobulk_ezkd_csv/rep1_14_16h': ['/Users/artemilin/Work/projects/Landscapes/scCandT/nanoCT_ezkd/fragments/emb14_16h_h3k27ac_1_fragments.tsv',
  '/Users/artemilin/Work/projects/Landscapes/scCandT/nanoCT_ezkd/fragments/emb14_16h_h3k27me3_1_fragments.tsv'],
 '/Users/artemilin/Work/projects/Landscapes/scCandT/pseudobulk_ezkd_csv/rep2_14_16h': ['/Users/artemilin/Work/projects/Landscapes/scCandT/nanoCT_ezkd/fragments/emb14_16h_h3k27ac_2_fragments.tsv',
  '/Users/artemilin/Work/projects/Landscapes/scCandT/nanoCT_ezkd/fragments/emb14_16h_h3k27me3_2_fragments.tsv']}